# 02 · Progressive distillation (Stage 2): 32 → 16 → 8 → 4 steps
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreyu20sharma/fastdiff-med/blob/main/notebooks/02_distillation.ipynb)

Stage 1 (run D) gets good samples in about 20 steps, but no ODE solver can follow the curved sampling
path in 4 steps. **Progressive distillation** (Salimans & Ho, 2022) *learns* the shortcut. In each
round a student, initialised from the teacher, learns to match **two** teacher DDIM steps with **one**
of its own:

1. The teacher takes two DDIM steps, x_t → x_t' → x_t''.
2. Solve for the x̃₀ that a single DDIM step t → t'' would need to land exactly on x_t''.
3. The student (v-prediction) is trained toward x̃₀ with loss weight max(SNR, 1).
4. The student becomes the next teacher.

In round 1 the teacher uses classifier-free guidance (w = 2), so guidance is **folded into the
student**: one network call per step instead of two. The 4-step student needs **4 calls per image,
compared with 2,000** for the guided 1000-step baseline.

Requires `checkpoints/D/ema.pt` from notebook 01. Each round is resumable: finished students are skipped.

### Setup
Works on **Google Colab** (saves to `MyDrive/fastdiff-med/`) and on **Kaggle** (saves to
`/kaggle/working/fastdiff-med/` and picks up uploaded checkpoints from `/kaggle/input/`). It clones or
updates the repo and installs dependencies. Training and distillation resume from their last
checkpoint, so after a disconnect you only rerun this notebook.

* `QUICK = True`: ~5-minute smoke test with tiny settings (outputs go to a separate `quick/` folder).
* `LITE = True`: about half the GPU time (fewer FID samples, shorter distillation rounds). The results
  are still valid, just a bit less precise.

*Colab: Runtime → Change runtime type → **T4 GPU**. Kaggle: Settings → Accelerator → **GPU T4**, Internet **on**.*

In [ ]:
import os, sys, glob, shutil
QUICK = False   # True = tiny smoke-test settings (~5 min)
LITE = True     # True = about half the GPU time (recommended if GPU time is limited)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_KAGGLE = os.path.exists('/kaggle/working')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE, CODE = '/content/drive/MyDrive/fastdiff-med', '/content/fastdiff-med'
elif IN_KAGGLE:
    BASE, CODE = '/kaggle/working/fastdiff-med', '/kaggle/working/fastdiff-med-code'
else:  # local Jupyter, started from the repo or its notebooks/ folder
    CODE = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
    BASE = os.path.join(CODE, 'workdir')

if IN_COLAB or IN_KAGGLE:
    if not os.path.exists(CODE):
        !git clone -q https://github.com/shreyu20sharma/fastdiff-med.git {CODE}
    %cd {CODE}
    !git pull -q
    !pip install -q -r requirements.txt
else:
    %cd {CODE}
if CODE not in sys.path:
    sys.path.insert(0, CODE)

ROOT = BASE + ('/quick' if QUICK else '')
CKPT, RESULTS, DOCS = f'{ROOT}/checkpoints', f'{ROOT}/results', f'{ROOT}/docs'
os.environ['MEDMNIST_ROOT'] = f'{BASE}/data'  # dataset cache (~20 MB)
for d in (CKPT, RESULTS, DOCS, os.environ['MEDMNIST_ROOT']):
    os.makedirs(d, exist_ok=True)

if IN_KAGGLE:  # copy checkpoints from an uploaded Kaggle dataset, keeping the A/ B/ D/ layout
    for f in glob.glob('/kaggle/input/**/*', recursive=True):
        rel = f.split('checkpoints/', 1)[-1] if 'checkpoints/' in f else None
        if rel and os.path.isfile(f) and not os.path.exists(f'{CKPT}/{rel}'):
            os.makedirs(os.path.dirname(f'{CKPT}/{rel}'), exist_ok=True)
            shutil.copy(f, f'{CKPT}/{rel}')
    print('checkpoints:', sorted(glob.glob(f'{CKPT}/**/*.pt', recursive=True)))

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (no GPU: only the light cells will work)')
print('Saving to', ROOT)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Image, display
def show(path, width=None):
    if os.path.exists(path):
        display(Image(filename=path, width=width))
    else:
        print('missing:', path)

In [ ]:
ITERS = 50 if QUICK else (1500 if LITE else 4000)   # iterations per distillation round
BATCH = 32 if QUICK else 128
END_STEPS = 4                           # set to 8 if short on time
!python -m src.distill --teacher {CKPT}/D/ema.pt --ckpt_dir {CKPT} --start_steps 32 --end_steps {END_STEPS} --iters_per_round {ITERS} --batch {BATCH} --log_every 500

## Visual check per round
The same seed and classes (one row per class) for the 20-step teacher with guidance and each
student. The student images should stay sharp and class-consistent as steps drop.

In [ ]:
import glob, matplotlib.pyplot as plt
from src.utils import load_model
from src.sampling import sample_images
from src.evaluate import grid_labels, save_grid

def grid_img(x):
    return ((x + 1) / 2).clamp(0, 1).view(7, 6, 3, 32, 32).permute(0, 3, 1, 4, 2).reshape(7 * 32, 6 * 32, 3).numpy()

panels = []
m, dif, _ = load_model(f'{CKPT}/D/ema.pt')
panels.append(('Teacher D · 20 steps · CFG (40 calls)', sample_images(m, dif, grid_labels(6), 'ddim', 20, 2.0, seed=0)))
students = sorted(glob.glob(f'{CKPT}/distill/student_*step.pt'), key=lambda p: -int(p.split('_')[-1][:-7]))
for p in students:
    m, dif, cfg = load_model(p)
    panels.append((f"Student · {cfg['steps']} steps ({cfg['steps']} calls)",
                   sample_images(m, dif, grid_labels(6), 'ddim', cfg['steps'], 1.0, seed=0)))
fig, axes = plt.subplots(1, len(panels), figsize=(3.2 * len(panels), 4), dpi=110)
for ax, (t, x) in zip(np.atleast_1d(axes), panels):
    ax.imshow(grid_img(x)); ax.set_title(t, fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Distillation loss per round
import pandas as pd
log = pd.read_csv(f'{CKPT}/distill/log.csv')
for (r, s), g in log.groupby(['round', 'student_steps']):
    plt.plot(g['iter'], g['loss'], label=f'round {r} → {s} steps')
plt.yscale('log'); plt.xlabel('iteration'); plt.ylabel('weighted x₀ loss'); plt.legend(); plt.show()